Here's a 4-dimensional, 125-point dataset with 4 material classes.

Columns: sample_id, density (g/cm³), youngs_modulus (GPa), thermal_conductivity (W/m·K), service_temp (°C, melting point for inorganics / degradation temp for polymers), and material_class as the ground-truth label. Classes are Metal (32), Ceramic (30), Polymer (33), Semiconductor (30).

Note that in real life, ground-truth labels will not be available for clustering tasks. In this practice problem they are given for learning purposes. 

the class-level ranges are loosely anchored to real materials intuition — polymers are the lightest and most compliant, ceramics the stiffest with the highest melting points, metals conduct heat well, semiconductors sit between metals and ceramics. Those directional relationships are physically genuine. But every individual row is a synthetic Gaussian draw around a class centroid, not a measured material, so no point corresponds to a real substance

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

# Step 1: 
Read the data file `materials_clustering.csv`, save it as `df` and check if it had been read properly


In [4]:
#sample_id is the unique identifier, and material_class is the cluster



# Step 2:
Explore the data
- Notice that only the numeric columns are described
- Check if the column counts are the same
- Check & compare the mean, min and max to see how comparable each column range is - (do they need scaling?)
- Check the min values - are there any invalid values (e.g. negative young's modulus) ?
- Check the max values - are there any possible incorrect values?

# Step 3:
Format the data for clustering.

Let's see if we can cluster these material classes correctly.
For that, we will ignore the ground truth (material_class) for now.
Also we don't need the sample_id for calculations - so let's drop both from the dataset and save it as a separate variable called `data`
Note that we will need the material_class - ground truth values later for comparison

# Step 4:

pick the columns of the dataset and save them as `features` - we will need them later.

print them out to see if that's correct

# Step 5:

Clustering

Let's build a K-means cluster model. Our number of clusters will be the number of categories in the ground truth values

#semiconductor, polymer, ceramic & metal

Don't forget to set a random_state for reproducibility

# Step 6:
This is a 4-D dataset, so it's difficult to visualize in a plot

And this is something we will have to get used to - as complex datasets often have high-dimensional data

But for now - let's do a pairplot() using seaborn library (You will have to read up documentation to see how to do this part):

#1. take the `kmeans.labels_` as type string and save it as a separate column in the data file called `cluster`- this gives us the clusters assigned to each datapoint.

#2. use seaborn.pairplot() to plot the data, - refer to the documentation to see what happens here.
#arguments to the function - vars = features, hue = "cluster", diag_kind="kde"
#later on you can swap `kde` with `hist` and see what happens

# Step 7:
You can simply run the below codes.
Some of these are advanced codes - so you might have to carefully go through line-by-line to understand them.
But if that's difficult at this stage - just try and understand the logic for now.


In [ ]:
#Let's see how many datapoints have been assigned to each cluster
print(data["cluster"].value_counts().sort_index())


At a glance it looks like some level of clustering has been done correctly, but since we have the ground truth values, let's check how good the clustering is.

 **STEP 7 (a):** 
 
 sorted side-by-side view (manual inspection)

 Sort by cluster so same-cluster rows sit together. Scroll through
 each block and write down, by eye, which material_class shows up
 most often in each cluster.

In [ ]:
view = pd.DataFrame({
    "sample_id": df["sample_id"],
    "material_class": df["material_class"],
    "cluster": kmeans.labels_
}).sort_values("cluster").reset_index(drop=True)
 
print(view.to_string())  # <- scroll through this whole table in the notebook

**STEP 7 (b):** 

same information, without scrolling 125 rows

groupby + value_counts collapses each cluster block to class counts.

This is exactly what you were counting by hand in Cell 2 — just faster.

In [ ]:
df.groupby(kmeans.labels_)["material_class"].value_counts()
 #Here you will see that there are some problems

**Step 7 (c):**

There seem to be some problems in the earlier assignment - can you see what they are?

we can use pandas function crosstab() to compare between each pair of original data and clustered data to see if the category matches


In [ ]:
mapping = (pd.crosstab(df["material_class"], kmeans.labels_)
             .idxmax(axis=0).to_dict())     # cluster id -> dominant true class
data["cluster_named"] = pd.Series(kmeans.labels_).map(mapping)
print(data["cluster_named"])

**STEP 7 (c):** 

formalize it — crosstab + idxmax

A crosstab lays out every (true class, cluster) count at once.

idxmax(axis=0) then reads off, for each cluster COLUMN, which

class ROW has the largest count -> your "eyeballed" mapping, done programmatically.


In [ ]:
ct = pd.crosstab(df["material_class"], kmeans.labels_)
print(ct)
 
mapping = ct.idxmax(axis=0).to_dict()
print("\nmapping (cluster id -> class name):", mapping)
 
df["predicted"] = pd.Series(kmeans.labels_).map(mapping)
df["match"] = df["predicted"] == df["material_class"]
print(f"\nagreement: {df['match'].mean():.1%}")
 
 

**Step 7 (d):**

BREAK IT: a run where idxmax fails

random_state=1 on this same unscaled data produces a case where
idxmax assigns the SAME class name to two different clusters —
because it decides column-by-column, with no rule stopping two
columns from both claiming the same "winner" row.
 


In [ ]:
kmeans_bad = KMeans(n_clusters=4, n_init='auto', random_state=1)
kmeans_bad.fit(data)
 
ct_bad = pd.crosstab(df["material_class"], kmeans_bad.labels_)
print(ct_bad)
 
mapping_bad = ct_bad.idxmax(axis=0).to_dict()
print("\nidxmax mapping:", mapping_bad)
print("distinct class names used:", sorted(set(mapping_bad.values())))
print("class name NEVER assigned to any cluster:",
      set(df["material_class"].unique()) - set(mapping_bad.values()))
 

**Step 7 (e)**

FIX IT: the Hungarian algorithm (linear_sum_assignment)

Instead of choosing each column's winner independently, solve for the ONE-TO-ONE matching between clusters and classes that maximizes
total agreement across ALL columns simultaneously. No class name can be used twice; none can be left out.


In [ ]:
from scipy.optimize import linear_sum_assignment
 
row_idx, col_idx = linear_sum_assignment(-ct_bad.values)   # negate: maximize, not minimize
mapping_fixed = {ct_bad.columns[c]: ct_bad.index[r] for r, c in zip(row_idx, col_idx)}
 
print("Hungarian mapping:", mapping_fixed)
print("distinct class names used:", sorted(set(mapping_fixed.values())),
      "-> all 4 present:", len(set(mapping_fixed.values())) == 4)
 
df["predicted_fixed"] = pd.Series(kmeans_bad.labels_).map(mapping_fixed)
df["match_fixed"] = df["predicted_fixed"] == df["material_class"]
print(f"agreement (Hungarian): {df['match_fixed'].mean():.1%}")

OK, now that's all settled - we know what our accuracy is when we run a kmeans clustering without scaling our data.

# Step 8

Let's do some scaling, 
And see how this compares

This should fix Misclassification due to the two axes being on two different scales
Let's use Standard Scaler

In [ ]:
scaler = StandardScaler()#Create an instance of the standard scaler   #Z-score, #Standard normal distribution
scaler.fit(data)       
# You will get an error here - what's the error about?

How do we trouble shoot something like this - if you notice what the error is complaining about - says a certain category cannot be scaled.

So let's check what our data file looks like,you can use `data.head()`

In [ ]:
data.head()


You can see that there are two columns (cluster & cluster_named) that you don't want to be scaling

#cluster_named has categorical values and you can't scale them - this is the cause of the error

So we need to remove them for now and save the rest as a separate variable `data_scaled`

In [ ]:
data_scaled = data.drop(columns=["cluster","cluster_named"])
data_scaled.head()

In [52]:
#Now let's scale them 
data_scaled = scaler.fit_transform(data_scaled)
#print(data_scaled)

# Step 9:

Now that the data is scaled, let's use that to fit a kmeans model

Since we already made one model called `kmeans` this new one with the scaled data should be named differently. Let's call it `kmeans_std`

# Step 10:

Let's compare the two clustering tasks - unscaled kmeans and scaled kmeans_std

We will look at`adjusted_rand_score` against material_class, since we have the ground truth available for this teaching dataset:

This is a clean, honest, unit-independent number — 1.0 means perfect agreement with the true classes, 0.0 means no better than random. 

So, it only looks at the two label assignments — cluster ID vs true class — and asks whether pairs of points are grouped consistently.

In [ ]:
from sklearn.metrics import adjusted_rand_score
print("unscaled ARI:", adjusted_rand_score(df["material_class"], kmeans.labels_))
print("scaled   ARI:", adjusted_rand_score(df["material_class"], kmeans_std.labels_))

In [ ]:
#Let's plot a confusion matrix to help visualize

def plot_confusion(true_labels, cluster_labels, title, ax):
    ct = pd.crosstab(true_labels, cluster_labels)
    # Hungarian reordering so each class lines up with its best-matching
    # cluster column -- makes the diagonal mean something
    row_idx, col_idx = linear_sum_assignment(-ct.values)
    ct = ct[[ct.columns[c] for c in col_idx]]
    sns.heatmap(ct, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(title)
    ax.set_ylabel("true material_class")
    ax.set_xlabel("kmeans cluster")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
plot_confusion(df["material_class"], kmeans.labels_, "Unscaled", axes[0])
plot_confusion(df["material_class"], kmeans_std.labels_, "Scaled (kmeans_std)", axes[1])
plt.tight_layout()

Once scaled, we can see that the materials categories are clustered very nicely! Except for **One Misclassification**.

It might be worth looking into why this particular datapoint was misclassified, but misclassifications are a part of machine learning - and we will talk more about them later on.

One thing to keep in mind is that we could compare how well we did our clustering based on the ground truth values. But in real-life problems clustering is an **unsupervised** problem, so we don't have anything to compare our performance against!

In such a case, how do we know if our clustering was done well?


A few practical checks, without ground truth:

1. Look at the cluster profiles. For each cluster, compute the mean of every feature. Do the numbers make physical sense? Does one cluster look like "light, soft, low melting point" (plausibly polymers) and another "dense, stiff, very high melting point" (plausibly ceramics)? Domain knowledge is doing the validating here, not a number.
2. Check stability. Rerun with a few different random_state values. If the groupings keep coming out the same, that's reassuring; if they keep changing, treat the result with suspicion.
3. There are label-free scores (silhouette is the common one) that check whether clusters are tight and well-separated. They can help compare a few options — but they can also be confidently wrong, as we saw earlier when they preferred the unscaled clustering. So treat them as one input, not the final word.

# Bottom line: without labels, "good clustering" is ultimately a judgement call informed by domain knowledge, not something a single metric can certify.




In [ ]:
# supporting code: cluster profiles (no labels used)
df["cluster"] = kmeans_std.labels_
df.groupby("cluster")[features].mean().round(1)